In [ ]:
%env CUDA_VISIBLE_DEVICES=1
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import os
from ne_aligned_utils.plot import plot_aligned_embedding
from ne_aligned_utils.data import make_joint_gmm
from ne_aligned_utils.utils import sparsify_affinities

from openTSNE import TSNE
from openTSNE.affinity import Affinities
import scipy.sparse as sp
from sklearn.decomposition import PCA


In [ ]:
dataset = "gmm"
tol = 1e-12
maxIter = int(1e4)
thetas = [1.0, 2.0, 3.0]

n0 = 600
n1 = 600
colors =  np.array(["r"]*n0 + ["b"]*n1)
seeds = [0, 1, 2, 3, 4]
seed = 0
eps_times_list = [1, 10, 100]


res_path = os.path.join("data", dataset)
fig_path = "data/figures"

In [ ]:
# general for NE experiments
init_str = "random"  # "pca" or "random"
init_scale = 1

batch_size = 2**14
negative_samples = 5
n_epochs = 200
loss_mode = "infonce"

In [ ]:
# 2d first
embd_dim = 2


In [ ]:
# sample non-sparsified

alignment = True
sample_pos = "per_point"
sparsify = None


In [ ]:

fig, ax = plt.subplots(len(thetas), len(eps_times_list), figsize=(6, 6))

for j, theta in enumerate(thetas):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_theta_{theta}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                            f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                            f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               )
        if i == 0:
            ax[i, j].set_title(f"theta {theta}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle(f"Non-sparsified embeddings GMM")
fig.savefig(os.path.join(fig_path, f"gmm_nonsparsified.png"))

In [ ]:
concordances_sample = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
rand_sample = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
for i, theta in enumerate(thetas):
    for j, eps_times in enumerate(eps_times_list):
        metrics_path = os.path.join(res_path,
                                    f"{dataset}_theta_{theta}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                    f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                    f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
        metrics = np.load(metrics_path)
        concordances_sample[i, j] = metrics["concordance"]
        rand_sample[i, j] = metrics["rand"]


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

for i, eps_times in enumerate(eps_times_list):
    ax[0].plot(thetas, rand_sample[:, i].mean(-1), label=f"eps/{eps_times}")
    ax[1].plot(thetas, concordances_sample[:, i].mean(-1), label=f"eps/{eps_times}")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[0].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")


fig.suptitle(f"Sampled NE metrics GMM 2D")
fig.savefig(os.path.join(fig_path, f"gmm_sample_metrics_2d.png"))

In [ ]:
# iterate over sparsified

alignment = True
sample_pos = None
sparsify = "topk"


In [ ]:

fig, ax = plt.subplots(len(thetas), len(eps_times_list), figsize=(6, 6))

for j, theta in enumerate(thetas):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_theta_{theta}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                            f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                            f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               )
        if i == 0:
            ax[i, j].set_title(f"theta {theta}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)

fig.suptitle(f"Top-k sparsified embeddings GMM")
fig.savefig(os.path.join(fig_path, f"gmm_topk.png"))

In [ ]:
concordances_topk = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
rand_topk = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
for i, theta in enumerate(thetas):
    for j, eps_times in enumerate(eps_times_list):
        metrics_path = os.path.join(res_path,
                                    f"{dataset}_theta_{theta}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                    f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                    f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
        metrics = np.load(metrics_path)
        concordances_topk[i, j] = metrics["concordance"]
        rand_topk[i, j] = metrics["rand"]




In [ ]:
rand_topk.shape

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

for i, eps_times in enumerate(eps_times_list):
    ax[0].plot(thetas, rand_topk[:, i].mean(-1), label=f"eps/{eps_times}")
    ax[1].plot(thetas, concordances_topk[:, i].mean(-1), label=f"eps/{eps_times}")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[0].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")


fig.suptitle(f"Top-k sparsified metrics GMM 2D")
fig.savefig(os.path.join(fig_path, f"gmm_topk_metrics_2d.png"))

In [ ]:
# EOT
sparsify = None # "topk" # None # "topk" # None


In [ ]:
t=0

fig, ax = plt.subplots( len(thetas), len(thetas), figsize=(6, 4))

for j, theta in enumerate(thetas):
    for i, eps_times in enumerate(eps_times_list):
        filename = os.path.join(res_path,
                            f"{dataset}_theta_{theta}_seed_{seed}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}.npz")
        a = np.load(os.path.join(res_path, filename))
        embd = a["embd"]
        plot_aligned_embedding(embd[::-1],
                               n0,
                               s=0.1,
                               color =colors[::-1],
                               shuffle=False,
                               ax=ax[i, j],
                               )
        if i == 0:
            ax[i, j].set_title(f"theta {theta}")
        if j == 0:
            ax[i, j].text(s=f"eps / {eps_times}", rotation="vertical", va="center", ha="right", x=-0.1, y=0.5, transform=ax[i, j].transAxes)
fig.suptitle(f"EOT{t} embeddings")
fig.savefig(os.path.join(fig_path, f"gmm_EOT{t}.png"))

In [ ]:
concordances_eot0 = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
rand_eot0 = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
for i, theta in enumerate(thetas):
    for j, eps_times in enumerate(eps_times_list):
        metrics_path = os.path.join(res_path,
                                    f"{dataset}_theta_{theta}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}_metrics.npz")
        metrics = np.load(metrics_path)
        concordances_eot0[i, j] = metrics["concordance"]
        rand_eot0[i, j] = metrics["rand"]


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

for i, eps_times in enumerate(eps_times_list):
    ax[0].plot(thetas, rand_eot0[:, i].mean(-1), label=f"eps/{eps_times}")
    ax[1].plot(thetas, concordances_eot0[:, i].mean(-1), label=f"topk eps/{eps_times}")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[0].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")

fig.suptitle(f"EOT{t} metrics GMM 2D")
fig.savefig(os.path.join(fig_path, f"gmm_EOT{t}_metrics_2d.png"))

In [ ]:
# comparison plot
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(thetas, rand_sample[:, -1].mean(-1), label=f"sampled ne", linestyle="dashed")
ax[0].plot(thetas, rand_topk[:, -1].mean(-1), label=f"top-k ne", linestyle="dotted")
ax[0].plot(thetas, rand_eot0[:, -1].mean(-1), label=f"EOT0 ", linestyle="solid")

ax[1].plot(thetas, concordances_sample[:, -1].mean(-1), label=f"sampled ne", linestyle="dashed")
ax[1].plot(thetas, concordances_topk[:, -1].mean(-1), label=f"top-k ne", linestyle="dotted")
ax[1].plot(thetas, concordances_eot0[:, -1].mean(-1), label=f"EOT0", linestyle="solid")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[1].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")

fig.suptitle(f"Comparison metrics GMM 2D")
fig.savefig(os.path.join(fig_path, f"gmm_comparison_metrics_2d.png"))

In [ ]:
# 6d
embd_dim = 6

In [ ]:
# sample non-sparsified

alignment = True
sample_pos = "per_point"
sparsify = None


In [ ]:
concordances_sample = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
rand_sample = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
for i, theta in enumerate(thetas):
    for j, eps_times in enumerate(eps_times_list):
        metrics_path = os.path.join(res_path,
                                    f"{dataset}_theta_{theta}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                    f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                    f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
        metrics = np.load(metrics_path)
        concordances_sample[i, j] = metrics["concordance"]
        rand_sample[i, j] = metrics["rand"]


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

for i, eps_times in enumerate(eps_times_list):
    ax[0].plot(thetas, rand_sample[:, i].mean(-1), label=f"eps/{eps_times}")
    ax[1].plot(thetas, concordances_sample[:, i].mean(-1), label=f"eps/{eps_times}")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[0].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")


fig.suptitle(f"Sampled NE metrics GMM 6D")
fig.savefig(os.path.join(fig_path, f"gmm_sample_metrics_6d.png"))

In [ ]:
# iterate over sparsified

alignment = True
sample_pos = None
sparsify = "topk"


In [ ]:
concordances_topk = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
rand_topk = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
for i, theta in enumerate(thetas):
    for j, eps_times in enumerate(eps_times_list):
        metrics_path = os.path.join(res_path,
                                    f"{dataset}_theta_{theta}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_init_{init_str}"+
                                    f"_scale_{init_scale}_bs_{batch_size}_m_{negative_samples}_epochs_{n_epochs}"+
                                    f"_loss_{loss_mode}_align_{alignment}_sample_pos_{sample_pos}_metrics.npz")
        metrics = np.load(metrics_path)
        concordances_topk[i, j] = metrics["concordance"]
        rand_topk[i, j] = metrics["rand"]



In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

for i, eps_times in enumerate(eps_times_list):
    ax[0].plot(thetas, rand_topk[:, i].mean(-1), label=f"rand eps/{eps_times}")
    ax[1].plot(thetas, concordances_topk[:, i].mean(-1), label=f"concordance eps/{eps_times}")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[0].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")


fig.suptitle(f"Top-k sparsified metrics GMM 6D")
fig.savefig(os.path.join(fig_path, f"gmm_topk_metrics_6d.png"))

In [ ]:
# EOT
sparsify = None # "topk" # None # "topk" # None
t = 0

In [ ]:
concordances_eot0 = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
rand_eot0 = np.zeros((len(thetas), len(eps_times_list), len(seeds)))
for i, theta in enumerate(thetas):
    for j, eps_times in enumerate(eps_times_list):
        metrics_path = os.path.join(res_path,
                                    f"{dataset}_theta_{theta}_epstimes_{eps_times}_sparsify_{sparsify}_dim_{embd_dim}_EOT_t_{t}_metrics.npz")
        metrics = np.load(metrics_path)
        concordances_eot0[i, j] = metrics["concordance"]
        rand_eot0[i, j] = metrics["rand"]


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))

for i, eps_times in enumerate(eps_times_list):
    ax[0].plot(thetas, rand_eot0[:, i].mean(-1), label=f"rand eps/{eps_times}")
    ax[1].plot(thetas, concordances_eot0[:, i].mean(-1), label=f"concordance eps/{eps_times}")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[0].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")

fig.suptitle(f"EOT{t} metrics GMM 6D")
fig.savefig(os.path.join(fig_path, f"gmm_EOT{t}_metrics_6d.png"))

In [ ]:
# compare metrics

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
ax[0].plot(thetas, rand_sample[:, -1].mean(-1), label=f"sampled ne eps / 100", linestyle="dashed")
ax[0].plot(thetas, rand_topk[:, 0].mean(-1), label=f"top-k ne eps/1", linestyle="dotted")
ax[0].plot(thetas, rand_eot0[:, 0].mean(-1), label=f"EOT0 eps/1", linestyle="solid")

ax[1].plot(thetas, concordances_sample[:, -1].mean(-1), label=f"sampled ne eps/100", linestyle="dashed")
ax[1].plot(thetas, concordances_topk[:, 0].mean(-1), label=f"top-k ne, eps/1", linestyle="dotted")
ax[1].plot(thetas, concordances_eot0[:, 0].mean(-1), label=f"EOT0, eps/1", linestyle="solid")

ax[0].set_xlabel("theta")
ax[0].set_ylabel("rand index")
ax[1].legend()
ax[1].set_xlabel("theta")
ax[1].set_ylabel("concordance")

fig.suptitle(f"Comparison metrics GMM 6D")
fig.savefig(os.path.join(fig_path, f"gmm_comparison_metrics_6d.png"))

In [ ]:
# test diag settings

In [ ]:
## 2D,


In [ ]:
theta = 3.0
eps_times = 1
seed = 0

In [ ]:
clean1, clean2, clusters0, clusters1, = make_joint_gmm(n0=n0, n1=n1, p=1000, q=6, theta=3.0,seed=0, clean=True)

In [ ]:
clusters = np.concatenate([clusters0, clusters1])

colorj = np.array([plt.get_cmap("tab10")(i) for i in clusters])
colors = np.array(colors)

In [ ]:
diag_tsne = np.load("data/gmm/gmm_theta_3_seed_0_epstimes_1_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*diag_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("cluster")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag_tsne[order].T, s=1, c=colors[order])
ax[1].set_title("batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.suptitle(f"GMM Diagonal tSNE, seed {seed}, theta {theta}, eps times {eps_times}")
fig.savefig(os.path.join(fig_path, f"gmm_seed_{seed}_theta_{theta}_diag_tsne_eps_times_{eps_times}.png"))

In [ ]:
# different seed for order of clusters
diag_tsne1 = np.load("data/gmm/gmm_theta_3_seed_1_epstimes_1_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*diag_tsne1[order].T, s=1, c=colorj[order])
ax[0].set_title("OT-diagonal tSNE on joint PCA50, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag_tsne1[order].T, s=1, c=colors[order])
ax[1].set_title("OT-diagonal tSNE on joint PCA50, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

In [ ]:
# no diagonal
no_diag_tsne = np.load("data/gmm/gmm_theta_3_seed_0_epstimes_1_diag_False_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*no_diag_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("Non-diagonal tSNE on joint PCA50, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*no_diag_tsne[order].T, s=1, c=colors[order])
ax[1].set_title("Non-diagonal tSNE on joint PCA50, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

In [ ]:
P = np.load("data/gmm/gmm_theta_3_seed_0_epstimes_1_diag_False_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["P_sparse"]

In [ ]:
P.min(), P.max()

In [ ]:
clusters0.shape

In [ ]:
sorted_idx0 = np.argsort(clusters0)
sorted_idx1 = np.argsort(clusters1)
sorted_idx = np.concatenate([sorted_idx0, sorted_idx1 + n0])

In [ ]:
P.shape

In [ ]:
plt.imshow(P[np.ix_(sorted_idx0, sorted_idx1)], interpolation="none")

In [ ]:
from ne_aligned_utils.utils import get_bipartite_graph

In [ ]:
g = get_bipartite_graph(P, diag=True).toarray()
#np.fill_diagonal(g, g.max())

In [ ]:
g_plot = g.copy()

np.fill_diagonal(g_plot, g_plot.max())
plt.imshow(g_plot[np.ix_(sorted_idx, sorted_idx)], interpolation="none")
plt.axis('off')
plt.colorbar()
plt.savefig(os.path.join(fig_path, f"gmm_full_affinity_matrix.png"), dpi=300)

In [ ]:
g[:n0, :n0].argsort(axis=1)[:, -15:].shape

In [ ]:
g_sparse = sparsify_affinities(g, n0, n1, k=15)

In [ ]:
g_sparse.sum(1)

In [ ]:
g_plot = g_sparse.copy()
np.fill_diagonal(g_plot, g_plot.max())
plt.imshow(g_plot[np.ix_(sorted_idx, sorted_idx)], interpolation="none")
plt.colorbar()
plt.axis('off')
plt.savefig(os.path.join(fig_path, f"gmm_diag_sparse_affinity_matrix.png"), dpi=300)

In [ ]:
g_sparse /= g_sparse.sum()



In [ ]:
embd_dim = 2
seed = 2
init = np.random.randn(n0+n1 , embd_dim)
#init = PCA(n_components=embd_dim).fit_transform(clean1)
init /= init.std(0)[0]
init /= 10000

In [ ]:
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=seed,
                early_exaggeration_iter=66,
                n_iter=133)
affinities = Affinities()
affinities.P = sp.csr_matrix(g_sparse)

embd_diag_sparse = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(10,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*embd_diag_sparse[order].T, s=1, c=colorj[order])
ax[0].set_title(f"cluster")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*embd_diag_sparse[order].T, s=1, c=colors[order])
ax[1].set_title(f"batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

fig.suptitle(f"{dataset} Diagonal sparse tSNE")
fig.savefig(os.path.join(fig_path, f"gmm_diag_sparse_tsne.png"))

In [ ]:
g_plot = g.copy()
np.fill_diagonal(g_plot, g_plot.max())
plt.imshow(g_plot[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")

In [ ]:
embd_dim = 2
seed = 2
#init = np.random.randn(n0 , embd_dim)
init = PCA(n_components=embd_dim).fit_transform(clean1)
init /= init.std(0)[0]
init /= 10000

In [ ]:
g_single = g[:n0, :n0]

g_single[g_single < 0] = 0

#g_single -= g_single[g_single > 0].min()

g_single /= g_single.sum()
print(g_single.min())

In [ ]:
g_single[np.ix_(sorted_idx0, sorted_idx0)][:100, :100].mean()

In [ ]:
g_single[np.ix_(sorted_idx0, sorted_idx0)][:100, 100:].mean()


In [ ]:
plt.hist(g_single[np.ix_(sorted_idx0, sorted_idx0)][:100, :100].flatten(), density=True)
plt.hist(g_single[np.ix_(sorted_idx0, sorted_idx0)][:100, 100:].flatten(), density=True)


In [ ]:
g_plot = g_single.copy()
np.fill_diagonal(g_plot, g_plot.max())
plt.imshow(g_plot[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")
plt.colorbar()

In [ ]:
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=seed,
                early_exaggeration_iter=66,
                n_iter=133)
affinities = Affinities()
affinities.P = sp.csr_matrix(g_single)

embd_single = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
g_single -= g_single[g_single > 0].min()
g_single[g_single< 0] = 0

g_single /= g_single.sum()


In [ ]:
g_plot = g_single.copy()
np.fill_diagonal(g_plot, g_plot.max())
plt.imshow(g_plot[np.ix_(sorted_idx0, sorted_idx0)], interpolation="none")
plt.colorbar()

In [ ]:
plt.hist(g_plot.flatten())

In [ ]:
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=seed,
                early_exaggeration_iter=66,
                n_iter=133)
affinities = Affinities()
affinities.P = sp.csr_matrix(g_single)

embd_min = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(n0)
print(order.dtype)

ax[0].scatter(*embd_single[order].T, s=1, c=colorj[order])
ax[0].set_title("t-SNE on single diagonal block")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*embd_min[order].T, s=1, c=colorj[order])
ax[1].set_title("t-SNE on single diagonal block with min subtracted")
ax[1].set_aspect("equal")
ax[1].axis("off")

In [ ]:
from sklearn.manifold import spectral_embedding

In [ ]:
spec_embd0 = spectral_embedding(g_single, n_components=embd_dim)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(15,5))

order = np.random.permutation(n0)
print(order.dtype)

ax.scatter(*spec_embd0[order].T, s=1, c=colorj[order])
ax.set_title("t-SNE on single diagonal block")
ax.set_aspect("equal")
ax.axis("off")

In [ ]:
## eps = 100

In [ ]:
diag100_tsne = np.load("data/gmm/gmm_theta_3_seed_0_epstimes_100_diag_True_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*diag100_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("OT-diagonal tSNE on joint PCA50 eps=100, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*diag100_tsne[order].T, s=1, c=colors[order])
ax[1].set_title("OT-diagonal tSNE on joint PCA50, eps=100, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

In [ ]:
no_diag100_tsne = np.load("data/gmm/gmm_theta_3_seed_0_epstimes_100_diag_False_sparsify_None_norm_whole_dim_2_init_random_scale_10000_epochs_200_tsne.npz")["embd"]


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*no_diag100_tsne[order].T, s=1, c=colorj[order])
ax[0].set_title("Non-diagonal tSNE on joint PCA50, eps=100, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")

ax[1].scatter(*no_diag100_tsne[order].T, s=1, c=colors[order])
ax[1].set_title("Non-diagonal tSNE on joint PCA50, eps=100, batch")
ax[1].set_aspect("equal")
ax[1].axis("off")

In [ ]:
# try near constant similarity
np.random.seed(seed=seed)
P = np.random.uniform(0, 1, size=600**2).reshape(600, 600)

P = 0.5 + P*0.01

np.fill_diagonal(P, 0)
P /= P.sum()

In [ ]:
P_plot = P
np.fill_diagonal(P_plot, P_plot.max())
plt.imshow(P, interpolation="none")

In [ ]:
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=seed,
                early_exaggeration_iter=66,
                n_iter=133)
affinities = Affinities()
affinities.P = sp.csr_matrix(P)

embd_random = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*embd_random.T, s=1, )
ax[0].set_title("Non-diagonal tSNE on joint PCA50, eps=100, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")


In [ ]:
import scipy.linalg

In [ ]:
# block diagonal

blocks = [0.6+ 0.01*np.random.randn(100, 100) for i in range(6)]

block_diag = scipy.linalg.block_diag(*blocks)
P =   np.random.randn(600, 600)*0.01+0.595
P[block_diag > 0] = block_diag[block_diag > 0]



In [ ]:
P[:100, :100].mean(), P[:100, 100:].mean()

In [ ]:
plt.imshow(P, interpolation="none")

In [ ]:
np.fill_diagonal(P, 0)
P /= P.sum()

In [ ]:
embedder = TSNE(n_components=embd_dim,
                verbose=True,
                random_state=seed,
                early_exaggeration_iter=66,
                n_iter=133)
affinities = Affinities()
affinities.P = sp.csr_matrix(P)

embd_block_random = embedder.fit(affinities=affinities, initialization=init)

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(15,5))

order = np.random.permutation(len(colorj))
print(order.dtype)

ax[0].scatter(*embd_block_random.T, s=1, )
ax[0].set_title("Non-diagonal tSNE on joint PCA50, eps=100, cell type")
ax[0].set_aspect("equal")
ax[0].axis("off")
